In [1]:
import os
os.chdir("../")

In [2]:
import pandas as pd
import mlflow
from src.dsproject.constants import *
from src.dsproject.utils.common import read_yaml, create_directories, save_json
from dataclasses import dataclass
from pathlib import Path
from dotenv import load_dotenv
from src.dsproject import logger

load_dotenv()

c:\Users\divya\Desktop\proj\ml-ops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
from dataclasses import dataclass

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    model_path: Path
    test_data_path: Path
    metric_file_name: Path
    all_params: dict
    target_column: str
    mlflow_uri: str

In [4]:
class ConfigurationManager:
    def __init__(self, config_file_path: Path = CONFIG_FILE_PATH, 
                 params_file_path: Path = PARAMS_FILE_PATH,
                 schema_file_path: Path = SCHEMA_FILE_PATH):
        
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        model_evaluation_config = self.config.model_evaluation
        
        model_evaluation_config_dir = Path(model_evaluation_config.root_dir)
        create_directories([model_evaluation_config_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=model_evaluation_config_dir,
            model_path=Path(model_evaluation_config.model_path),
            test_data_path=Path(model_evaluation_config.test_data_path),
            metric_file_name=Path(model_evaluation_config.metric_file_name),
            all_params=self.params,
            target_column=self.schema.TARGET_COLUMN.name,
            mlflow_uri=os.getenv("MLFLOW_URI")
        )

        return model_evaluation_config

In [5]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import mlflow.sklearn
from urllib.parse import urlparse
import numpy as np
import joblib

In [6]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate_model(self, actual, predicted):
        rmse = np.sqrt(mean_squared_error(actual, predicted))
        r2 = r2_score(actual, predicted)
        mae = mean_absolute_error(actual, predicted)
        return rmse, r2, mae

    def log_metrics_to_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_X = test_data.drop(columns=[self.config.target_column])
        test_y = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted = model.predict(test_X)
            rmse, r2, mae = self.evaluate_model(test_y, predicted)

            scores = {
                "rmse": rmse,
                "r2": r2,
                "mae": mae
            }

            save_json(path = Path(self.config.metric_file_name), data = scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

        if tracking_url_type_store != "file":
            mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticNetModel")
        else:
            mlflow.sklearn.log_model(model, "model")



In [7]:
try:
    config_manager = ConfigurationManager()
    model_evaluation_config = config_manager.get_model_evaluation_config()
    logger.info(f"Model Evaluation Config: {model_evaluation_config}")
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_metrics_to_mlflow()
except Exception as e:
    logger.exception(f"Error in getting Model Evaluation Config: {e}")
    raise e

[2026-05-06 14:49:56,456: INFO: common: Reading YAML file from: config\config.yaml]
[2026-05-06 14:49:56,463: INFO: common: Successfully read YAML file: config\config.yaml]
[2026-05-06 14:49:56,465: INFO: common: Reading YAML file from: params.yaml]
[2026-05-06 14:49:56,469: INFO: common: Successfully read YAML file: params.yaml]
[2026-05-06 14:49:56,471: INFO: common: Reading YAML file from: schema.yaml]
[2026-05-06 14:49:56,475: INFO: common: Successfully read YAML file: schema.yaml]
[2026-05-06 14:49:56,477: INFO: common: Creating directory at: artifacts]
[2026-05-06 14:49:56,480: INFO: common: Successfully created directory: artifacts]
[2026-05-06 14:49:56,483: INFO: common: Creating directory at: artifacts\model_evaluation]
[2026-05-06 14:49:56,486: INFO: common: Successfully created directory: artifacts\model_evaluation]
[2026-05-06 14:49:56,488: INFO: 99147925: Model Evaluation Config: ModelEvaluationConfig(root_dir=WindowsPath('artifacts/model_evaluation'), model_path=WindowsPa

2026/05/06 14:50:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 14:50:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/06 14:50:19 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Successfully registered model 'ElasticNetModel'.
2026/05/06 14:50:24 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticNetModel, version 1
Created version '1' of model 'ElasticNetModel'.
